# QuetHoaDon · Chấm test lần 2: cấu hình sản phẩm (bản web)

Quy tắc đặt trước khi chạy: [`docs/reports/final_test_protocol.md`](../docs/reports/final_test_protocol.md). Tóm tắt:
- Cấu hình sản phẩm = cấu hình đóng băng (cùng model, cùng OCR) + luật mặt hàng 19/11 + tổng tiền từ các món + chặn ảnh
  khó đọc. Ảnh bị chặn tính là **không đọc được** (bất lợi cho mình).
- **Hai cổng** phải qua trước: cell 5 (tính lại số đóng băng 17/11, phải KHỚP) và cell 6 (chấm offline = đầu ra của app,
  phải PASS). Cell 7 tự dừng nếu một cổng chưa qua.
- Cell 7 chấm **một lần**; kết quả được công bố dù tốt hay xấu.

Chạy trên Kaggle: **GPU T4**, **Internet On**, gắn 3 dataset: **Vietnamese Receipts MC_OCR 2021**, `phase1-result`,
`quethoadon` (không bắt buộc). Cách chạy: **Save Version → Save & Run All** (chạy nền, lưu output). Khoảng 20–30 phút.

In [ ]:
# CELL 1 - Cài detectron2 (LayoutXLM cần, 5-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn nhánh main và cài thư viện. Mong đợi: một dòng commit, "xong"
BRANCH = "main"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb 2>&1 | tail -2
print("xong")

In [ ]:
# CELL 3 [KIỂM THỬ] - GPU, dữ liệu, checkpoint, giao thức có trong mã nguồn, unit test.
# Mong đợi: "GPU: True Tesla T4", mọi đường dẫn có giá trị, "giao thức: có", "Ran ... tests" rồi "OK"
import glob, os, sys, json, csv, subprocess, torch
REPO = '/kaggle/working/QuetHoaDon'
sys.path.insert(0, REPO)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
one = lambda pattern: sorted(glob.glob(pattern, recursive=True))[-1]
CSV = one('/kaggle/input/**/mcocr_train_df.csv')
IMG_RAW = os.path.dirname(one('/kaggle/input/**/train_images/**/*.jpg'))
TOKC = os.path.dirname(one('/kaggle/input/**/phase1_eval/tok_ft_n05/test.jsonl'))
CK3 = os.path.dirname(one('/kaggle/input/**/m1b_v3_checkpoint/best/config.json'))
for k in ("CSV", "IMG_RAW", "TOKC", "CK3"):
    print(f"{k} = {globals()[k]}")
COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)
print("giao thức:", "có" if os.path.exists(f'{REPO}/docs/reports/final_test_protocol.md') else "THIẾU - dừng lại")
W = '/kaggle/working'
os.makedirs(f'{W}/eval', exist_ok=True)
!cd {REPO} && python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

In [ ]:
# CELL 4 [KIỂM THỬ] - File chia tập và ảnh tiền xử lý của test (CPU, vài phút, có thanh tiến trình).
# Mong đợi: "test: 116 hóa đơn, thiếu ảnh 0"
import cv2
from tqdm.auto import tqdm
from src.common.config import load_config
from src.preprocessing.image_preprocessor import ImagePreprocessor
from src.training.data import load_jsonl
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out {W}/dataset > {W}/dataset_summary.txt
with open(f'{W}/dataset/mcocr_manifest_split.csv', encoding='utf-8') as f:
    split_rows = list(csv.DictReader(f))
IMG_PRE = f'{W}/img_pre'
os.makedirs(IMG_PRE, exist_ok=True)
pp = ImagePreprocessor(**load_config(f'{REPO}/configs')['model']['preprocessing'])
test_ids = sorted({r['img_id'] for r in split_rows if r['split'] == 'test'})
for img_id in tqdm(test_ids, desc="tiền xử lý"):
    out = os.path.join(IMG_PRE, os.path.splitext(img_id)[0] + '.jpg')
    if not os.path.exists(out):
        cv2.imwrite(out, pp.process(os.path.join(IMG_RAW, img_id)).image, [cv2.IMWRITE_JPEG_QUALITY, 95])
recs = load_jsonl(os.path.join(TOKC, 'test.jsonl'))
print(f"test: {len(recs)} hóa đơn, thiếu ảnh {sum(not os.path.exists(os.path.join(IMG_PRE, r['image'])) for r in recs)}")

In [ ]:
# CELL 5 [KIỂM THỬ] CỔNG 1 - Tính lại cấu hình đóng băng trên test: phải ra đúng số đã báo cáo 17/11 (GPU, vài phút).
# Mong đợi: "cổng 1 ... KHỚP"
def strict_counts(path):
    a = json.load(open(path))['app']
    f = a['fields']
    return {"SELLER": f['SELLER']['correct'], "ADDRESS": f['ADDRESS']['correct'], "date": f['TIMESTAMP']['correct'],
            "time": f['TIMESTAMP']['time']['correct'], "TOTAL": f['TOTAL_COST']['correct'],
            "all": a['receipts_all_correct']['correct']}

!python scripts/evaluate_layoutxlm.py --checkpoint "{CK3}" --token-dir {TOKC} --image-dir {IMG_PRE} --split test --allow-test --save-probs {W}/eval/m1b_v3_test_probs.npz --out {W}/eval/m1b_v3_test.json 2>&1 | tail -2
!python scripts/decode_from_probs.py --token-dir {TOKC} --split test --allow-test --probs {W}/eval/m1b_v3_test_probs.npz --datetime --cleanup --out {W}/eval/decoded_frozen_test.json > /dev/null
got = strict_counts(f'{W}/eval/decoded_frozen_test.json')
GATE1 = got == {"SELLER": 77, "ADDRESS": 29, "date": 89, "time": 61, "TOTAL": 71, "all": 21}
print("cổng 1, cấu hình đóng băng:", got, "->", "KHỚP" if GATE1 else "LỆCH: DỪNG, không chạy cell 7")

In [ ]:
# CELL 6 [KIỂM THỬ] CỔNG 2 - Chấm offline có cho đúng đầu ra của app (web) không, trên test (CPU, vài giây).
# Không đọc nhãn nên không chấm gì. Mong đợi: "identical 116" và "PASS"
!python scripts/verify_product_path.py --token-dir {TOKC} --split test --allow-test --probs {W}/eval/m1b_v3_test_probs.npz --out {W}/eval/verify_product_test.json
GATE2 = json.load(open(f'{W}/eval/verify_product_test.json'))['pass']
print("cổng 2:", "PASS" if GATE2 else "FAIL: DỪNG, không chạy cell 7")

In [ ]:
# CELL 7 - CHẤM MỘT LẦN cấu hình sản phẩm trên test (CPU, vài giây). Chỉ chạy khi hai cổng đã qua.
# In: bảng đúng tuyệt đối, bảng ghép cặp với cấu hình đóng băng, mặt hàng, số ảnh bị chặn. Kết quả nào cũng được công bố.
assert GATE1 and GATE2, "Một cổng chưa qua: không chấm. Xem cell 5 / 6."
!python scripts/decode_from_probs.py --token-dir {TOKC} --split test --allow-test --probs {W}/eval/m1b_v3_test_probs.npz --datetime --cleanup --total-from-items --quality-check --out {W}/eval/decoded_product_test.json | grep -E -A10 "Strict reading|totals taken|flagged"
!python scripts/compare_strict.py --token-dir {TOKC} --split test --allow-test --a frozen={W}/eval/decoded_frozen_test.json --b product={W}/eval/decoded_product_test.json --out {W}/eval/paired_product_test.json
!python scripts/score_items.py --token-dir {TOKC} --split test --allow-test --quality-check --out {W}/eval/items_product_test.json | sed -n 1,13p

# Phụ: độ đúng trên các ảnh không bị chặn (không phải chỉ số chính)
from src.evaluation.app_metrics import app_scores
from src.evaluation.reproduction import reference_predictions
prod = json.load(open(f'{W}/eval/decoded_product_test.json'))
flagged = set(prod['unreadable'] or [])
gts = {r['img_id']: r['gt'] for r in recs if r['img_id'] not in flagged}
sub = app_scores(gts, reference_predictions(prod))
print(f"phụ: bỏ {len(flagged)} ảnh bị chặn ->", {k: f"{v['correct']} / {v['n']}" for k, v in sub['fields'].items()},
      "| cả bốn:", f"{sub['receipts_all_correct']['correct']} / {sub['receipts_all_correct']['n']}")
json.dump({"commit": COMMIT, "gate1": GATE1, "gate2": GATE2, "flagged": sorted(flagged),
           "frozen": got, "product": strict_counts(f'{W}/eval/decoded_product_test.json')},
          open(f'{W}/eval/final_test_summary.json', 'w'), indent=2)
print("tóm tắt:", json.load(open(f'{W}/eval/final_test_summary.json')))

In [ ]:
# CELL 8 - Nén kết quả để tải về (tab Output)
!cd /kaggle/working && zip -q -r final_test.zip eval && ls -la final_test.zip